# Phase 10 (Helper) — Export KPIs to CSV for Power BI

**Project:** Banking Transaction Monitoring & Fraud Analytics Platform

Power BI can connect to MySQL directly, but that needs an extra connector installed.
The simplest, most reliable path for a first dashboard is to **export the KPI views to
CSV** and import those into Power BI. This notebook writes one CSV per dashboard dataset
into a `powerbi/` folder.

(You can still use the live MySQL connection later — see the Phase 10 build guide.)

## 1. Configuration

In [ ]:
DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": "YOUR_MYSQL_PASSWORD",   # <-- your MySQL password
    "database": "federal_bank",
}
OUTPUT_DIR = "powerbi"   # CSVs for Power BI will be written here

## 2. Imports & connect

In [ ]:
import os
import pandas as pd
import mysql.connector

conn = mysql.connector.connect(**DB_CONFIG)
print("Connected to database:", DB_CONFIG["database"])

## 3. The datasets to export
The nine KPI views plus a few fraud-analytics queries for the Fraud page.

In [ ]:
EXPORTS = {
    # --- KPI views (Phase 9) ---
    "kpi_overview":          "SELECT * FROM v_kpi_overview",
    "monthly_trend":         "SELECT * FROM v_monthly_trend",
    "monthly_growth":        "SELECT * FROM v_monthly_growth",
    "branch_performance":    "SELECT * FROM v_branch_performance",
    "top_cities":            "SELECT * FROM v_top_cities",
    "channel_popularity":    "SELECT * FROM v_channel_popularity",
    "customer_value":        "SELECT * FROM v_customer_value",
    "customer_segmentation": "SELECT * FROM v_customer_segmentation",
    "customer_risk":         "SELECT * FROM v_customer_risk",
    # --- Fraud-analytics datasets (for the Fraud page) ---
    "fraud_by_rule":  "SELECT rule_triggered, COUNT(*) AS alerts "
                      "FROM fraud_alerts GROUP BY rule_triggered ORDER BY alerts DESC",
    "fraud_by_risk":  "SELECT risk_level, COUNT(*) AS alerts "
                      "FROM fraud_alerts GROUP BY risk_level",
    "fraud_by_branch": "SELECT b.branch_name, b.city, COUNT(*) AS alerts "
                       "FROM fraud_alerts fa "
                       "JOIN transactions t ON fa.transaction_id = t.transaction_id "
                       "JOIN accounts a     ON t.account_id      = a.account_id "
                       "JOIN branches b     ON a.branch_id       = b.branch_id "
                       "GROUP BY b.branch_name, b.city ORDER BY alerts DESC",
    "fraud_trend":   "SELECT YEAR(alert_timestamp) AS year, MONTH(alert_timestamp) AS month, "
                     "MONTHNAME(alert_timestamp) AS month_name, COUNT(*) AS alerts "
                     "FROM fraud_alerts GROUP BY year, month, month_name ORDER BY year, month",
    "alerts_detail": "SELECT alert_id, transaction_id, customer_id, rule_triggered, risk_level, "
                     "status, alert_reason, alert_timestamp FROM fraud_alerts ORDER BY alert_id",
}

## 4. Export everything to CSV

In [ ]:
def export_all(conn, exports, out_dir):
    # Run each query and save the result as a CSV in out_dir. Returns (name, rows) list.
    os.makedirs(out_dir, exist_ok=True)
    cur = conn.cursor()
    summary = []
    for name, query in exports.items():
        cur.execute(query)
        cols = [d[0] for d in cur.description]
        df = pd.DataFrame(cur.fetchall(), columns=cols)
        path = os.path.join(out_dir, f"{name}.csv")
        df.to_csv(path, index=False)
        summary.append((name, len(df)))
        print(f"  {name}.csv  ({len(df)} rows)")
    cur.close()
    return summary

summary = export_all(conn, EXPORTS, OUTPUT_DIR)
print(f"\nExported {len(summary)} CSV files to '{OUTPUT_DIR}/'.")

## 5. Close

In [ ]:
conn.close()
print("Done. Import these CSVs into Power BI (see the Phase 10 build guide).")